# Can we predict next-month churn?
Target: `churn_next_month`. Each row is an account-month snapshot. Plan:
1. Inspect data and check for leakage
2. Build features available at snapshot time only
3. Validate with a **time-based split** (train on earlier months, test on later ones)
4. Compare against baselines and report

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); print(df.shape); df.head()

In [ ]:
print(df.isna().sum().sum(),'missing;',df.duplicated(['account_ref','snapshot_month']).sum(),'duplicate account-months')
print(df.account_ref.nunique(),'accounts,',df.snapshot_month.min(),'to',df.snapshot_month.max())
print('churn rate:',df.churn_next_month.mean().round(3))
df.describe().T

## 1. Leakage check: `lifetime_months`
The dictionary says it is the *total months the account was active, from a CRM export of 2026-01-05* — measured **after** the snapshot, so it encodes the future. Compare it with `tenure_months`:

In [ ]:
df['gap']=df.lifetime_months-df.tenure_months
print(df.groupby('churn_next_month').gap.describe())
print('Share of rows with lifetime==tenure, by label:'); print((df.gap==0).groupby(df.churn_next_month).mean())

For churners `lifetime_months == tenure_months` in every row (the account ended right after the snapshot); for non-churners it is always larger. It reveals the label but **is not available at prediction time**, so I exclude it. Quantifying how misleading it would be:

In [ ]:
print('AUC if we (wrongly) used the leak:',roc_auc_score(df.churn_next_month,(df.gap==0).astype(int)))

## 2. Churn over time
Churn rate is far from stable — it spikes in late 2024 / Jan 2025. This matters for validation.

In [ ]:
df=df.sort_values(['account_ref','snapshot_month']).reset_index(drop=True)
m=df.groupby('snapshot_month').churn_next_month.agg(['mean','size'])
fig,ax=plt.subplots(figsize=(9,3.5)); ax.plot(m.index,m['mean'],marker='o'); ax.set_ylabel('churn rate'); plt.xticks(rotation=60); ax.set_title('Churn rate by snapshot month'); plt.show()
print(df.groupby('plan').churn_next_month.mean().round(3))
print(df.groupby(pd.cut(df.tenure_months,[0,1,3,6,12,24]),observed=True).churn_next_month.mean().round(3))

## 3. Features
Only information known at the snapshot: current-month values plus lagged values from the same account's *previous* snapshots. `snapshot_month` is not used as a feature (a calendar effect can't be extrapolated to future months).

In [ ]:
g=df.groupby('account_ref')
df['fee_per_seat']=df.monthly_fee/df.seats
df['usage_per_seat']=df.usage_hours/df.seats
df['usage_change']=df.usage_hours-g.usage_hours.shift(1)
df['usage_vs_own_avg']=df.usage_hours-g.usage_hours.transform(lambda s:s.shift(1).expanding().mean())
df['tickets_prev3']=g.support_tickets.transform(lambda s:s.shift(1).rolling(3,min_periods=1).sum())
df['seats_change']=df.seats-g.seats.shift(1)
df['plan_change']=((df.plan!=g.plan.shift(1))&(g.cumcount()>0)).astype(int)
cat=['plan']
num=['seats','monthly_fee','tenure_months','usage_hours','support_tickets','fee_per_seat','usage_per_seat','usage_change','usage_vs_own_avg','tickets_prev3','seats_change','plan_change']
feats=cat+num; y='churn_next_month'

## 4. Time-based split
Train on snapshots ≤ 2025-03, test on 2025-04 onward. Test months are strictly later than training months; a random split would mix months and be optimistic given the drift above.

In [ ]:
tr=df[df.snapshot_month<='2025-03']; te=df[df.snapshot_month>='2025-04']
print(len(tr),len(te),'| churn rate train/test:',tr[y].mean().round(3),te[y].mean().round(3))
print('accounts appearing in both:',len(set(tr.account_ref)&set(te.account_ref)))

In [ ]:
pre=ColumnTransformer([('c',OneHotEncoder(),cat),('n',make_pipeline(SimpleImputer(strategy='constant',fill_value=0),StandardScaler()),num)])
models={
 'logistic':make_pipeline(pre,LogisticRegression(max_iter=2000,class_weight='balanced')),
 'grad. boosting':make_pipeline(ColumnTransformer([('c',OneHotEncoder(),cat)],remainder='passthrough'),HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)),
}
res={}
for n,mod in models.items():
    mod.fit(tr[feats],tr[y]); p=mod.predict_proba(te[feats])[:,1]; res[n]=p
    print(f'{n:15s} AUC={roc_auc_score(te[y],p):.3f}  PR-AUC={average_precision_score(te[y],p):.3f}')
print('prevalence baseline PR-AUC:',te[y].mean().round(3),'(AUC 0.5)')
for f in ['tenure_months','usage_hours','usage_per_seat']:
    print(f'single feature -{f}: AUC', round(roc_auc_score(te[y],-te[f]),3))

## 5. Practical value: who to target?

In [ ]:
best=max(res,key=lambda n:roc_auc_score(te[y],res[n])); p=res[best]; print('best model:',best)
t=te.assign(p=p).sort_values('p',ascending=False)
for q in [.05,.10,.20,.30]:
    k=int(len(t)*q); print(f'top {q:.0%} flagged: precision={t[y].iloc[:k].mean():.2f}  recall={t[y].iloc[:k].sum()/t[y].sum():.2f}  lift={t[y].iloc[:k].mean()/t[y].mean():.1f}x')
pr,rc,_=precision_recall_curve(te[y],p); plt.plot(rc,pr); plt.axhline(te[y].mean(),ls='--',c='gray'); plt.xlabel('recall'); plt.ylabel('precision'); plt.title('Precision-recall (test)'); plt.show()

In [ ]:
# calibration by score quintile (matters if scores are read as probabilities; logistic uses class_weight so is miscalibrated by design)
b=pd.qcut(p,5,duplicates='drop'); print(pd.DataFrame({'pred':p,'obs':te[y].values}).groupby(b,observed=True).mean().round(3))
imp=permutation_importance(models[best],te[feats],te[y],scoring='roc_auc',n_repeats=5,random_state=0)
pd.Series(imp.importances_mean,feats).sort_values().plot.barh(title='Permutation importance (AUC drop)'); plt.show()

In [ ]:
# robustness: several train/test cut points
for cut in ['2024-09','2025-01','2025-05','2025-08']:
    a=df[df.snapshot_month<cut]; b_=df[df.snapshot_month>=cut]
    m_=make_pipeline(ColumnTransformer([('c',OneHotEncoder(),cat)],remainder='passthrough'),HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)).fit(a[feats],a[y])
    print(f'train<{cut} test>={cut}: AUC={roc_auc_score(b_[y],m_.predict_proba(b_[feats])[:,1]):.3f} (test churn {b_[y].mean():.2f})')

## Conclusions (from the run at time of writing)
- **Leakage:** `lifetime_months` equals `tenure_months` for every churner and exceeds it for every non-churner (AUC 1.0 on its own). It is measured after the snapshot, so it was excluded. Any model that includes it will look perfect and be useless.
- **Predictability is weak-to-moderate.** On the later test window (2025-04+): logistic regression AUC ≈ 0.67, PR-AUC ≈ 0.19 (prevalence 0.096); gradient boosting is no better (AUC ≈ 0.66). Low usage is the strongest single signal (AUC ≈ 0.61); plan and tenure add little.
- **Usable for prioritising, not for confident calls:** the top 10% of scores catch ~25% of churners at ~24% precision (2.5x lift); the top 30% catch ~53%. Most churn is not flagged.
- **Calibration:** the logistic model uses `class_weight='balanced'`, so its scores are rankings, not probabilities (predicted ~0.3–0.7 vs observed 0.05–0.20). Recalibrate before using as probabilities.
- **Instability:** churn rate swings from ~3% to ~55% by month (spike Nov 2024–Jan 2025) and is not explained by these features. AUC varies 0.61–0.70 across cut points. Training on the spike period and testing after it also shifts the base rate (19% → 10%).
- **Caveats:** accounts repeat across months (352 appear in both train and test); the final month per account may be censored; confirm every feature is available at scoring time. The next step with most upside is finding out what drove the late-2024 spike (pricing change? migration?) and adding that information.